# 02 — Validate Metrics

**Project:** AdIntel AI  
**Milestone:** 1 — Synthetic Ads Dataset Validation

Notebook ini melakukan validasi:

- Row count
- Primary key
- Foreign key
- Metric sanity check
- Scenario validation: revenue stable, impression naik, viewability turun, VCR turun


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
ISSUE_START_DATE = pd.Timestamp("2026-03-15")

print("Data dir:", DATA_DIR)


Data dir: d:\08. Portofolio\04. Github\portfolio\ai\adintel-ai\notebooks\data\synthetic


## 1. Load Data


In [ ]:
files = {
    "advertisers": "advertisers.csv",
    "campaigns": "campaigns.csv",
    "ad_groups": "ad_groups.csv",
    "creatives": "creatives.csv",
    "placements": "placements.csv",
    "inventory": "inventory.csv",
    "markets": "markets.csv",
    "devices": "devices.csv",
    "daily_ad_performance": "daily_ad_performance.csv",
    "video_performance": "video_performance.csv",
    "conversion_events": "conversion_events.csv",
    "billing_revenue": "billing_revenue.csv",
    "data_quality_logs": "data_quality_logs.csv",
}

data = {}
for table, filename in files.items():
    path = DATA_DIR / filename
    assert path.exists(), f"File tidak ditemukan: {path}"
    data[table] = pd.read_csv(path)

print("Loaded tables:", len(data))


AssertionError: File tidak ditemukan: d:\08. Portofolio\04. Github\portfolio\ai\adintel-ai\notebooks\data\synthetic\advertisers.csv

## 2. Row Count Check


In [ ]:
row_counts = pd.DataFrame([
    {"table": table, "rows": len(df), "columns": df.shape[1]}
    for table, df in data.items()
]).sort_values("table").reset_index(drop=True)

row_counts


In [ ]:
minimum_rows = {
    "advertisers": 20,
    "campaigns": 50,
    "ad_groups": 120,
    "creatives": 200,
    "placements": 50,
    "inventory": 200_000,
    "daily_ad_performance": 50_000,
    "video_performance": 10_000,
}

row_validation = []
for table, min_row in minimum_rows.items():
    actual = len(data[table])
    row_validation.append({
        "table": table,
        "min_expected": min_row,
        "actual": actual,
        "status": "PASS" if actual >= min_row else "FAIL"
    })

row_validation_df = pd.DataFrame(row_validation)
row_validation_df


## 3. Primary Key Validation


In [ ]:
primary_keys = {
    "advertisers": "advertiser_id",
    "campaigns": "campaign_id",
    "ad_groups": "ad_group_id",
    "creatives": "creative_id",
    "placements": "placement_id",
    "inventory": "inventory_id",
    "markets": "market_id",
    "devices": "device_id",
    "daily_ad_performance": "performance_id",
    "video_performance": "video_performance_id",
    "conversion_events": "conversion_event_id",
    "billing_revenue": "billing_id",
    "data_quality_logs": "dq_log_id",
}

pk_results = []

for table, pk in primary_keys.items():
    df = data[table]
    missing = df[pk].isna().sum()
    duplicate = df[pk].duplicated().sum()
    pk_results.append({
        "table": table,
        "primary_key": pk,
        "missing_pk": missing,
        "duplicate_pk": duplicate,
        "status": "PASS" if missing == 0 and duplicate == 0 else "FAIL"
    })

pk_results_df = pd.DataFrame(pk_results)
pk_results_df


## 4. Foreign Key Validation


In [ ]:
fk_checks = [
    ("campaigns", "advertiser_id", "advertisers", "advertiser_id"),
    ("ad_groups", "campaign_id", "campaigns", "campaign_id"),
    ("creatives", "advertiser_id", "advertisers", "advertiser_id"),
    ("daily_ad_performance", "advertiser_id", "advertisers", "advertiser_id"),
    ("daily_ad_performance", "campaign_id", "campaigns", "campaign_id"),
    ("daily_ad_performance", "ad_group_id", "ad_groups", "ad_group_id"),
    ("daily_ad_performance", "creative_id", "creatives", "creative_id"),
    ("daily_ad_performance", "placement_id", "placements", "placement_id"),
    ("daily_ad_performance", "market_id", "markets", "market_id"),
    ("daily_ad_performance", "device_id", "devices", "device_id"),
    ("video_performance", "performance_id", "daily_ad_performance", "performance_id"),
    ("conversion_events", "performance_id", "daily_ad_performance", "performance_id"),
    ("billing_revenue", "advertiser_id", "advertisers", "advertiser_id"),
    ("billing_revenue", "campaign_id", "campaigns", "campaign_id"),
    ("billing_revenue", "market_id", "markets", "market_id"),
    ("inventory", "placement_id", "placements", "placement_id"),
    ("inventory", "market_id", "markets", "market_id"),
    ("inventory", "device_id", "devices", "device_id"),
]

fk_results = []

for child_table, child_key, parent_table, parent_key in fk_checks:
    child_values = set(data[child_table][child_key].dropna().unique())
    parent_values = set(data[parent_table][parent_key].dropna().unique())
    missing_refs = child_values - parent_values

    fk_results.append({
        "child_table": child_table,
        "child_key": child_key,
        "parent_table": parent_table,
        "parent_key": parent_key,
        "missing_reference_count": len(missing_refs),
        "status": "PASS" if len(missing_refs) == 0 else "FAIL"
    })

fk_results_df = pd.DataFrame(fk_results)
fk_results_df


## 5. Metric Sanity Check


In [ ]:
perf = data["daily_ad_performance"].copy()
video = data["video_performance"].copy()
inv = data["inventory"].copy()
conv = data["conversion_events"].copy()

sanity_checks = {
    "negative impressions": (perf["impressions"] < 0).sum(),
    "clicks greater than impressions": (perf["clicks"] > perf["impressions"]).sum(),
    "negative spend": (perf["spend_usd"] < 0).sum(),
    "negative revenue": (perf["publisher_revenue_usd"] < 0).sum(),
    "viewable greater than measurable": (perf["viewable_impressions"] > perf["measurable_impressions"]).sum(),
    "viewability outside 0-1": ((perf["viewability_rate"] < 0) | (perf["viewability_rate"] > 1)).sum(),
    "video completes greater than starts": (video["video_completes"] > video["video_starts"]).sum(),
    "vcr outside 0-1": ((video["video_completion_rate"] < 0) | (video["video_completion_rate"] > 1)).sum(),
    "fill rate outside 0-1": ((inv["fill_rate"] < 0) | (inv["fill_rate"] > 1)).sum(),
    "win rate outside 0-1": ((inv["win_rate"] < 0) | (inv["win_rate"] > 1)).sum(),
    "negative conversions": (conv["conversions"] < 0).sum(),
    "negative conversion value": (conv["conversion_value_usd"] < 0).sum(),
}

sanity_df = pd.DataFrame([
    {"check": k, "issue_count": v, "status": "PASS" if v == 0 else "FAIL"}
    for k, v in sanity_checks.items()
])

sanity_df


## 6. Build Pre vs Post Period


In [ ]:
perf["date"] = pd.to_datetime(perf["date"])
video["date"] = pd.to_datetime(video["date"])
inv["date"] = pd.to_datetime(inv["date"])

perf["period"] = np.where(perf["date"] < ISSUE_START_DATE, "pre", "post")
video["period"] = np.where(video["date"] < ISSUE_START_DATE, "pre", "post")
inv["period"] = np.where(inv["date"] < ISSUE_START_DATE, "pre", "post")

perf["period"].value_counts()


## 7. Overall Performance Metric Validation


In [ ]:
perf_summary = (
    perf.groupby("period", as_index=False)
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spend_usd=("spend_usd", "sum"),
        publisher_revenue_usd=("publisher_revenue_usd", "sum"),
        measurable_impressions=("measurable_impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
    )
)

perf_summary["ctr"] = perf_summary["clicks"] / perf_summary["impressions"]
perf_summary["cpc"] = perf_summary["spend_usd"] / perf_summary["clicks"]
perf_summary["cpm"] = perf_summary["spend_usd"] / perf_summary["impressions"] * 1000
perf_summary["ecpm"] = perf_summary["publisher_revenue_usd"] / perf_summary["impressions"] * 1000
perf_summary["viewability"] = perf_summary["viewable_impressions"] / perf_summary["measurable_impressions"]

perf_summary


In [ ]:
def pct_change_from_summary(df, metric):
    pre = df.loc[df["period"] == "pre", metric].iloc[0]
    post = df.loc[df["period"] == "post", metric].iloc[0]
    return post / pre - 1

movement = pd.DataFrame([
    {"metric": "impressions", "change": pct_change_from_summary(perf_summary, "impressions")},
    {"metric": "publisher_revenue_usd", "change": pct_change_from_summary(perf_summary, "publisher_revenue_usd")},
    {"metric": "spend_usd", "change": pct_change_from_summary(perf_summary, "spend_usd")},
    {"metric": "cpm", "change": pct_change_from_summary(perf_summary, "cpm")},
    {"metric": "ecpm", "change": pct_change_from_summary(perf_summary, "ecpm")},
    {"metric": "viewability", "change": pct_change_from_summary(perf_summary, "viewability")},
])

movement["change_pct"] = movement["change"].map(lambda x: f"{x:.2%}")
movement


## 8. Video Metric Validation


In [ ]:
video_summary = (
    video.groupby("period", as_index=False)
    .agg(
        video_starts=("video_starts", "sum"),
        video_completes=("video_completes", "sum"),
    )
)

video_summary["vcr"] = video_summary["video_completes"] / video_summary["video_starts"]
video_summary


In [ ]:
vcr_change = pct_change_from_summary(video_summary, "vcr")
print(f"VCR change post vs pre: {vcr_change:.2%}")


## 9. Mix Shift Validation


In [ ]:
placements = data["placements"].copy()
devices = data["devices"].copy()

perf_enriched = (
    perf.merge(placements[["placement_id", "quality_tier", "inventory_type"]], on="placement_id", how="left")
        .merge(devices[["device_id", "platform"]], on="device_id", how="left")
)

quality_mix = (
    perf_enriched.groupby(["period", "quality_tier"], as_index=False)
    .agg(impressions=("impressions", "sum"))
)
quality_mix["share"] = quality_mix.groupby("period")["impressions"].transform(lambda x: x / x.sum())

quality_mix


In [ ]:
platform_mix = (
    perf_enriched.groupby(["period", "platform"], as_index=False)
    .agg(impressions=("impressions", "sum"))
)
platform_mix["share"] = platform_mix.groupby("period")["impressions"].transform(lambda x: x / x.sum())

platform_mix


## 10. Inventory Validation


In [ ]:
inv_summary = (
    inv.groupby("period", as_index=False)
    .agg(
        ad_requests=("ad_requests", "sum"),
        bid_requests=("bid_requests", "sum"),
        won_impressions=("won_impressions", "sum"),
        avg_inventory_quality_score=("inventory_quality_score", "mean"),
    )
)

inv_summary["fill_rate"] = inv_summary["won_impressions"] / inv_summary["ad_requests"]
inv_summary["win_rate"] = inv_summary["won_impressions"] / inv_summary["bid_requests"]

inv_summary


## 11. Final Validation Status


In [ ]:
all_pk_pass = (pk_results_df["status"] == "PASS").all()
all_fk_pass = (fk_results_df["status"] == "PASS").all()
all_sanity_pass = (sanity_df["status"] == "PASS").all()

impression_change = pct_change_from_summary(perf_summary, "impressions")
revenue_change = pct_change_from_summary(perf_summary, "publisher_revenue_usd")
viewability_change = pct_change_from_summary(perf_summary, "viewability")
vcr_change = pct_change_from_summary(video_summary, "vcr")

scenario_checks = pd.DataFrame([
    {
        "check": "Impression increased post issue",
        "value": impression_change,
        "status": "PASS" if impression_change > 0.10 else "CHECK"
    },
    {
        "check": "Revenue broadly stable",
        "value": revenue_change,
        "status": "PASS" if -0.15 <= revenue_change <= 0.20 else "CHECK"
    },
    {
        "check": "Viewability dropped around target range",
        "value": viewability_change,
        "status": "PASS" if -0.30 <= viewability_change <= -0.10 else "CHECK"
    },
    {
        "check": "VCR dropped around target range",
        "value": vcr_change,
        "status": "PASS" if -0.30 <= vcr_change <= -0.08 else "CHECK"
    },
])

print("PK validation:", "PASS" if all_pk_pass else "FAIL")
print("FK validation:", "PASS" if all_fk_pass else "FAIL")
print("Metric sanity:", "PASS" if all_sanity_pass else "FAIL")

scenario_checks
